<h3><b style="color:red">IMPORTANT:</b> Restart your session everytime you open this workshop</h3>

In [29]:
# Run the following install and restart your session.
%pip install -U evaluate rouge_score rouge torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


# Fine Tuning Large Language Model - Model

In this workshop, you will learn how to fine tune the prompts and the LLMs to enhance and improves its response.

In [12]:
# Import libraries
import torch, time
import pandas as pd
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, GenerationConfig, TrainingArguments

from peft import PeftModel, LoraConfig, get_peft_model, TaskType

In [13]:
# TODO: Load and explore the following datasets

dataset_name = "knkarthick/dialogsum"
model_name = "google/flan-t5-base"

# load the dataset
dataset = load_dataset(dataset_name)


In [5]:
# TODO: Print a record
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})


In [6]:
idx = 500
for k, v in dataset['train'][idx].items():
   print(f"{k}: {v}")

id: train_500
dialogue: #Person1#: What is your great struck?
#Person2#: I think I'm very good at planning. I manage my time perfectly, so that I can always get things done on time.
#Person1#: Why do you think you be a good fit or this job?
#Person2#: Because I'm very competent in this area.
#Person1#: Can you working under pressure?
#Person2#: Yes, I work well under pressure and find it dimly. However, I believe planning and probable management of my time can reduce tight deadlines.
#Person1#: Are you more a follower or a leader?
#Person2#: I don't try to go ahead of people and lead them, I'd rather cooperate with everybody else and get the job done by working together.
summary: #Person1# is interviewing #Person2#. #Person2# is good at planning, works well under pressure, and is a cooperator.
topic: interview


## Fine tuning the LLM model

In this workshop we will be turning the <code>google/flan-t5-base</code> model.

In [7]:
# Utility function to dump a model's tunable parameters

def print_trainable_model_params(model):
   trainable_model_params = 0
   all_model_params = 0
   for _, param in model.named_parameters():
      all_model_params += param.numel()
      if param.requires_grad:
         trainable_model_params += param.numel()
   return f"Trainable parameters: {trainable_model_params}\nTotal parameters: {all_model_params}\nPercentable of trainable parameters: {100 * trainable_model_params / all_model_params:.2f}%"

In [16]:
# TODO: Load T5 model
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [11]:
# TODO: Print number of trainable parameters
print(print_trainable_model_params(original_model))

Trainable parameters: 247577856
Total parameters: 247577856
Percentable of trainable parameters: 100.00%


### Preprocess the dialogue dataset

We will train the model to summarize dialogue by creating a dialogue-summary pair for the LLM to process. The dialogue is the training data and the summary is the label. This is supervized learning.

The prompt will be as follows

```
Summarize the following dialogue.\n
\n
Fred: ...\n
Barney: ...\n
\n
Summary:\n
Summary of the conversation between Fred and Barney
```

The prompt and the summary will be tokenized for the LLM

In [17]:
# Utitlity function to prepare the data for training
# Tokenize function
def tokenize_fn(data):
   start_prompt = 'Summarize the following dialogue.\n\n'
   end_prompt = '\n\nSummary:'
   prompt = [ start_prompt + d + end_prompt for d in data['dialogue'] ]
   summary = data['summary']
   data['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt").input_ids
   data['labels'] = tokenizer(summary, padding="max_length", truncation=True, return_tensors="pt").input_ids
   return data


In [18]:
# TODO: prepare the data for training with the tokenize_fn function
tokenized_dataset = dataset.map(tokenize_fn, batched=True)



Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [19]:
# TODO: Verify prepared data
idx = 500
for k, v in tokenized_dataset['train'][idx].items():
   print(f"{k}: {v}")


id: train_500
dialogue: #Person1#: What is your great struck?
#Person2#: I think I'm very good at planning. I manage my time perfectly, so that I can always get things done on time.
#Person1#: Why do you think you be a good fit or this job?
#Person2#: Because I'm very competent in this area.
#Person1#: Can you working under pressure?
#Person2#: Yes, I work well under pressure and find it dimly. However, I believe planning and probable management of my time can reduce tight deadlines.
#Person1#: Are you more a follower or a leader?
#Person2#: I don't try to go ahead of people and lead them, I'd rather cooperate with everybody else and get the job done by working together.
summary: #Person1# is interviewing #Person2#. #Person2# is good at planning, works well under pressure, and is a cooperator.
topic: interview
input_ids: [12198, 1635, 1737, 8, 826, 7478, 5, 1713, 345, 13515, 536, 4663, 10, 363, 19, 39, 248, 10056, 58, 1713, 345, 13515, 357, 4663, 10, 27, 317, 27, 31, 51, 182, 207, 44, 

In [20]:
print(tokenizer.decode(tokenized_dataset['train'][idx]['input_ids']))
print('---------------------------------------------')
print(tokenizer.decode(tokenized_dataset['train'][idx]['labels']))

Summarize the following dialogue. #Person1#: What is your great struck? #Person2#: I think I'm very good at planning. I manage my time perfectly, so that I can always get things done on time. #Person1#: Why do you think you be a good fit or this job? #Person2#: Because I'm very competent in this area. #Person1#: Can you working under pressure? #Person2#: Yes, I work well under pressure and find it dimly. However, I believe planning and probable management of my time can reduce tight deadlines. #Person1#: Are you more a follower or a leader? #Person2#: I don't try to go ahead of people and lead them, I'd rather cooperate with everybody else and get the job done by working together. Summary:</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pa

In [23]:
print(len(tokenizer.decode(tokenized_dataset['train'][idx]['input_ids'])))

2292


In [24]:
# TODO: Remove id, dialogue, summary and topic columns from dataset. We only want input_ids and labels
tokenized_dataset = tokenized_dataset.remove_columns(['id', 'dialogue', 'summary', 'topic'])


In [25]:
# TODO: Verify dataset again
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1500
    })
})


### Tune model with pre-processed dataset

We will use [<code>Trainer</code>](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#api-reference%20][%20transformers.Trainer) to train the original model. The training result will be written out. The trainer will be configure with [<code>TrainingArgument</code>](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments)

In [26]:
# CUDA information

print('CUDA available: ', torch.cuda.is_available())
if torch.cuda.is_available():
   print('B16 supported: ', torch.cuda.is_bf16_supported())
   torch.cuda.set_device(0)
   print('Current device: ', torch.cuda.current_device())
   print('CUDA device name: ', torch.cuda.get_device_name(0))

CUDA available:  False


## Fine tuning the LLM Model with Low-Rank Adaptation (LoRA) / Parameter Efficient Fine Tuning (PEFT)

We will add a LoRA adapter to the LLM (flan-t5-base) and train the adapter. The original LLM will be frozen. The adapter can be combined with the original LLM during inferencing.

In [27]:
# TODO: Configure LoRA
lora_config = LoraConfig(
    r = 8, # rank 8 => mx8 . 8xn
    lora_alpha = 16,
    target_modules = ['q', 'v' ],
    lora_dropout = 0.05,
    bias = 'none',
    task_type = TaskType.SEQ_2_SEQ_LM,
)


In [30]:
# TODO: Add LoRA to the LLM model to be trained
# load the base model - flan-t5
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
# load the LoRA adapter on the base model
lora_model = get_peft_model(base_model, lora_config)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [31]:
print(lora_model)

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
              

In [32]:
# TODO: Print number of parameters, compare LoRA to the original model
print(print_trainable_model_params(lora_model))


Trainable parameters: 884736
Total parameters: 248462592
Percentable of trainable parameters: 0.36%


In [33]:
# TODO: Train model with LoRA
output_dir = f"model-{str(int(time.time()))}"
lora_train_args = TrainingArguments(
    output_dir = output_dir,
    auto_find_batch_size = True,
    learning_rate = 1e-3,
    num_train_epochs = 1,
)

In [ ]:
# TODO: Create trainer and train model
trainer = Trainer(
    model = lora_model,
    args = lora_train_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['validation'],
)

# start the training
#trainer.train()

# save the model and the correspoonding tokenizer
#my_model = "chukmunnlee/chuk-t5-base-instruct"
#trainer.save_model(my_model)
#tokenizer.save_pretrained(my_model)


### Use a trained LoRA model

The training will take a few hours and over many iterations.

For the purpose of this workshop we use a save model [intotheverse/peft-dialogue-summary-checkpoint](https://huggingface.co/intotheverse/peft-dialogue-summary-checkpoint).

In [34]:
#TODO: Load the original model and add the pre-trained LoRA adaptation to the model
# This is the trained LoRA model
peft_dialogue_summary_checkpoint = 'intotheverse/peft-dialogue-summary-checkpoint'

# Load LoRA model
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
# Load the LoRA adapter on the base model
lora_model = PeftModel.from_pretrained(base_model, peft_dialogue_summary_checkpoint)

# Original model
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


adapter_config.json:   0%|          | 0.00/334 [00:00<?, ?B/s]

adapter_model.bin: reconstructing file:   0%|          |  0.00B / 14.2MB            

adapter_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## Evaluate LoRA model

In [ ]:
# TODO: Evaluate LoRA model against the original



In [35]:
# Prepare data for evaluation
dialogues = []
summaries = []
orig_model_summaries = []
lora_model_summaries = []
config = GenerationConfig(max_new_tokens=200)

for i in range(5):
   print(f'i = {i}')
   d = dataset['test'][i]['dialogue']
   s = dataset['test'][i]['summary']
   prompt = f"Summarize the following conversation.\n\n{d}\n\nSummary:"
   tokenized_prompt = tokenizer(prompt, return_tensors='pt').input_ids
   orig_resp = original_model.generate(input_ids=tokenized_prompt, generation_config=config)
   orig_resp_text = tokenizer.decode(orig_resp[0], skip_special_tokens=True)
   lora_resp = lora_model.generate(input_ids=tokenized_prompt, generation_config=config)
   lora_resp_text = tokenizer.decode(lora_resp[0], skip_special_tokens=True)

   summaries.append(s)
   orig_model_summaries.append(orig_resp_text)
   lora_model_summaries.append(lora_resp_text)

zipped_summaries = list(zip(summaries, orig_model_summaries, lora_model_summaries))
df = pd.DataFrame(zipped_summaries, columns=['label', 'original_model_summary', 'lora_model_summary'])
df

i = 0
i = 1
i = 2
i = 3
i = 4


,label,original_model_summary,lora_model_summary
0,Ms. Dawson helps #Person1# to write a memo to ...,The memo is to be distributed to all employees...,#Person1# asks Ms. Dawson to take a dictation ...
1,In order to prevent employees from wasting tim...,The memo is to be distributed to all employees...,#Person1# asks Ms. Dawson to take a dictation ...
2,Ms. Dawson takes a dictation for #Person1# abo...,The memo is to be distributed to all employees...,#Person1# asks Ms. Dawson to take a dictation ...
3,#Person2# arrives late because of traffic jam....,The traffic jam at the Carrefour intersection ...,#Person2# got stuck in traffic and #Person1# s...
4,#Person2# decides to follow #Person1#'s sugges...,The traffic jam at the Carrefour intersection ...,#Person2# got stuck in traffic and #Person1# s...


### Evaluate models with ROUGE/Bleu metrics

Recall-Oriented Understudy for Gisting Evaluate ([ROUGE](https://pub.aimind.so/unveiling-the-power-of-rouge-metrics-in-nlp-b6d3f96d3363)) is a set of metrics used to evaluate the quality of machine-generated text, such as summaries and translations. ROUGE metrics compare the generated text to a human-written reference and measure the overlap between the two.

The metrics range between 0 and 1, with higher scores indicating higher similarity between the baseline and generated text.

In [36]:
# TODO: create ROUGE metrics
rouge = evaluate.load('rouge')


In [39]:
# TODO: Evaluate the original model's result
orig_model_result = rouge.compute(
    predictions = orig_model_summaries,
    references = summaries,
    use_aggregator = True,
    use_stemmer = True,
)

lora_model_result = rouge.compute(
    predictions = lora_model_summaries,
    references = summaries,
    use_aggregator = True,
    use_stemmer = True,
)

print('Original model result:')
print(orig_model_result)
print('------------------------------')
print('LoRA model result:')
print(lora_model_result)


Original model result:
{'rouge1': np.float64(0.1692307692307692), 'rouge2': np.float64(0.03820816864295125), 'rougeL': np.float64(0.13364468864468865), 'rougeLsum': np.float64(0.13364468864468865)}
------------------------------
LoRA model result:
{'rouge1': np.float64(0.3408045927322088), 'rouge2': np.float64(0.10249242018904939), 'rougeL': np.float64(0.27097500453912726), 'rougeLsum': np.float64(0.27097500453912726)}


In [40]:
# TODO: Evaluate with Bleu metrics
bleu = evaluate.load('bleu')

In [42]:
orig_model_result = bleu.compute(
    predictions = orig_model_summaries,
    references = summaries,
)

lora_model_result = bleu.compute(
    predictions = lora_model_summaries,
    references = summaries,
)


print('Original model result:')
print(orig_model_result)
print('------------------------------')
print('LoRA model result:')
print(lora_model_result)


Original model result:
{'bleu': 0.0156413976950709, 'precisions': [0.2835820895522388, 0.08064516129032258, 0.03508771929824561, 0.019230769230769232], 'brevity_penalty': 0.24955905423103703, 'length_ratio': 0.41875, 'translation_length': 67, 'reference_length': 160}
------------------------------
LoRA model result:
{'bleu': 0.0819270536975234, 'precisions': [0.2747603833865815, 0.12337662337662338, 0.066006600660066, 0.020134228187919462], 'brevity_penalty': 1.0, 'length_ratio': 1.95625, 'translation_length': 313, 'reference_length': 160}
